Посилання на датасет: https://www.kaggle.com/datasets/shub99/sentiment-analysis-data

In [1]:
import pandas as pd
from sklearn.datasets import load_files
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.pipeline import make_pipeline
import nltk
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec

## EDA

Читаємо дата сет з відгуками 

In [2]:
df = pd.read_csv('Restaurant_Reviews.tsv', sep='\t')

Розмір даних

Перші 5 рядків даних.

1 - хороший вігук

0- негативний відгук

In [3]:
print(df.shape)
print(df.head(5))

(1000, 2)
                                              Review  Liked
0                           Wow... Loved this place.      1
1                                 Crust is not good.      0
2          Not tasty and the texture was just nasty.      0
3  Stopped by during the late May bank holiday of...      1
4  The selection on the menu was great and so wer...      1


Проаналізуємо скільки в дата сеті негативних та позитивних відгуків

In [4]:
df['Liked'].value_counts()

Liked
1    500
0    500
Name: count, dtype: int64

Як бачимо, негативних та позитивних відгуків порівну (500 позитивних та 500 негативних відгуків), що дозволяє використовувати метрику Accuracy як адекватний показник якості моделей

Розглянемо приклади негативних та позитивних відгуків.

Позитивні

In [5]:
for r in df[df['Liked'] == 1]['Review'].head(5):
    print(r)

Wow... Loved this place.
Stopped by during the late May bank holiday off Rick Steve recommendation and loved it.
The selection on the menu was great and so were the prices.
The fries were great too.
A great touch.


Негативні

In [6]:
for r in df[df['Liked'] == 0]['Review'].head(5):
    print(r)

Crust is not good.
Not tasty and the texture was just nasty.
Now I am getting angry and I want my damn pho.
Honeslty it didn't taste THAT fresh.)
The potatoes were like rubber and you could tell they had been made up ahead of time being kept under a warmer.


Розділяємо дата сет на тренувальний та тестовий

In [7]:
train_df, test_df = train_test_split(df, test_size=0.25, random_state=42)

In [8]:
x_train, y_train = train_df['Review'], train_df['Liked'] 
x_test, y_test = test_df['Review'], test_df['Liked'] 


In [9]:
print(f"Кількість для навчання: {len(x_train)}")
print(f"Кількість для тесту: {len(x_test)}")

Кількість для навчання: 750
Кількість для тесту: 250


In [10]:
print("Samples per class (training): {}".format(np.bincount(y_train)))

Samples per class (training): [372 378]


### Bag-of-word

In [11]:
vect = CountVectorizer().fit(x_train)
Xx_train = vect.transform(x_train)
print('x_train: \n{}'.format(repr(Xx_train)))

x_train: 
<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 7387 stored elements and shape (750, 1728)>


In [12]:
print("Розмір отриманої матриці:", Xx_train.shape)

Розмір отриманої матриці: (750, 1728)


In [13]:
print("Samples per class: {}".format(np.bincount(y_train)))

Samples per class: [372 378]


In [14]:
print("Vocabulary size: {}".format(len(vect.vocabulary_)))
print("Vocabulary content:\n {}".format(vect.vocabulary_))

Vocabulary size: 1728
Vocabulary content:
 {'the': 1508, 'turkey': 1573, 'and': 53, 'roast': 1267, 'beef': 138, 'were': 1667, 'bland': 170, 'spend': 1411, 'your': 1722, 'money': 975, 'time': 1533, 'some': 1391, 'place': 1138, 'else': 485, 'great': 658, 'atmosphere': 92, 'friendly': 608, 'fast': 547, 'service': 1336, 'if': 771, 'you': 1721, 'stay': 1430, 'in': 781, 'vegas': 1607, 'must': 995, 'get': 631, 'breakfast': 195, 'here': 725, 'at': 90, 'least': 849, 'once': 1046, 'is': 804, 'perfect': 1116, 'family': 540, 'nice': 1014, 'to': 1537, 'see': 1321, 'couple': 335, 'of': 1033, 'months': 977, 'later': 845, 'returned': 1251, 'had': 683, 'an': 52, 'amazing': 47, 'meal': 937, 'point': 1150, 'finger': 568, 'any': 57, 'item': 807, 'on': 1045, 'menu': 952, 'order': 1057, 'it': 806, 'won': 1694, 'be': 129, 'disappointed': 417, 'there': 1514, 'really': 1217, 'nothing': 1026, 'for': 592, 'me': 936, 'postinos': 1160, 'hope': 747, 'experience': 525, 'better': 152, 'every': 512, 'eat': 472, 'carin

In [15]:
feature_names = vect.get_feature_names_out()
print("Number of features: {}".format(len(feature_names)))
print("First 20 features:\n{}".format(feature_names[:20]))
print("Features 210 to 230:\n{}".format(feature_names[210:230]))
print("Every 750th feature:\n{}".format(feature_names[::750]))

Number of features: 1728
First 20 features:
['10' '100' '12' '17' '1979' '20' '2007' '30' '30s' '35' '40' '40min' '45'
 '4ths' '5lb' '70' '85' '90' 'about' 'above']
Features 210 to 230:
['buldogis' 'bunch' 'burger' 'burgers' 'burrittos' 'bus' 'business'
 'bussell' 'busy' 'but' 'butter' 'by' 'bye' 'caballero' 'caesar' 'cafe'
 'café' 'calamari' 'call' 'calligraphy']
Every 750th feature:
['10' 'horrible' 'tepid']


In [16]:
scores = cross_val_score(LogisticRegression(), Xx_train, y_train, cv=4)
print("Mean cross-validation accuracy: {:.2f}".format(np.mean(scores)))

Mean cross-validation accuracy: 0.82


### Навчання логістичної регресії з підбором гіперпараметрів

Для класифікації тональності (позитивний/негативний відгук) ми використовуємо **Logistic Regression**. Для досягнення найкращої точності застосовується GridSearchCV, який дозволяє автоматично знайти оптимальний параметр регуляризації C.

* **Крос-валідація (cv=5)**: розбиття даних на 5 частин для об'єктивної оцінки стабільності моделі.
* **Параметр C**: контроль складності моделі для запобігання перенавчанню.

In [17]:
param_grid = {'C': [0.001, 0.01, 0.1, 4, 3, 5]}
grid = GridSearchCV(LogisticRegression(), param_grid, cv=4)
grid.fit(Xx_train, y_train)
print("Best cross-validation score: {:.2f}".format(grid.best_score_))
print("Best parameters: ", grid.best_params_)

Best cross-validation score: 0.82
Best parameters:  {'C': 5}


In [18]:
xx_test = vect.transform(x_test)
print("{:.2f}".format(grid.score(xx_test, y_test)))

0.82


Stopwords

In [19]:
print('Number of stop words: {}'.format(len(ENGLISH_STOP_WORDS)))
print("Every 10th stopword:\n{}".format(list(ENGLISH_STOP_WORDS)[::10]))

Number of stop words: 318
Every 10th stopword:
['from', 'fill', 'of', 'show', 'alone', 'found', 'nothing', 'still', 'further', 'ever', 'not', 'here', 'see', 'amount', 'otherwise', 'yours', 'below', 'so', 'whence', 'else', 'hence', 'other', 're', 'about', 'they', 'hers', 'mill', 'sixty', 'me', 'several', 'noone', 'describe']


In [20]:
Vect = CountVectorizer(min_df=5, stop_words='english').fit(x_train)
XX_train = Vect.transform(x_train)
print('X_train with stop words:\n{}'.format(repr(x_train)))

X_train with stop words:
82                 The turkey and roast beef were bland.
991           Spend your money and time some place else.
789         Great atmosphere, friendly and fast service.
894    If you stay in Vegas you must get breakfast he...
398    Service is perfect and the family atmosphere i...
                             ...                        
106    The food was delicious, our bartender was atte...
270        The Veggitarian platter is out of this world!
860    This place is pretty good, nice little vibe in...
435    It was a huge awkward 1.5lb piece of cow that ...
102    Best Buffet in town, for the price you cannot ...
Name: Review, Length: 750, dtype: str


In [21]:
from sklearn.metrics import classification_report

y_pred_bow = grid.predict(xx_test)
print("BoW")
print(classification_report(y_test, y_pred_bow))

BoW
              precision    recall  f1-score   support

           0       0.81      0.84      0.82       128
           1       0.83      0.79      0.81       122

    accuracy                           0.82       250
   macro avg       0.82      0.82      0.82       250
weighted avg       0.82      0.82      0.82       250



In [22]:
grid = GridSearchCV(LogisticRegression(), param_grid, cv=4)
grid.fit(XX_train, y_train)
print("Best cross-validation score: {:.2f}".format(grid.best_score_))

Best cross-validation score: 0.77


### Аналіз результатів моделі BoW

Отримана точність на крос-валідації становить **0.82** (або 82%). Це свідчить про те, що навіть простий метод підрахунку слів дозволяє ефективно розділяти позитивні та негативні відгуки.

**Основні спостереження:**
* Застосування стоп-слів та порогу мінімальної частоти (min_df=5) зменшило розмір словника, що на даному датасеті призвело до незначного зниження точності порівняно з базовою моделлю BoW.
* Модель стабільно працює на невеликому датасеті (1000 відгуків).

Далі цей результат буде використано як **baseline** (базову точку) для порівняння з більш складними методами, такими як TF-IDF та Word2Vec.

### TF-IDF (term frequency-inverse document frequency)

In [23]:
pipe = make_pipeline(TfidfVectorizer(min_df=5, norm=None), LogisticRegression())
param_grid = {'logisticregression__C': [0.001, 0.01, 0.1, 1, 10]}
grid = GridSearchCV(pipe, param_grid, cv=5)
grid.fit(x_train, y_train)
print(grid.best_params_)
print("Best cross-validation score: {:.2f}".format(grid.best_score_))

{'logisticregression__C': 0.01}
Best cross-validation score: 0.76


In [24]:
vectorizer = grid.best_estimator_.named_steps["tfidfvectorizer"]
X_Train = vectorizer.transform(x_train)
max_value = X_Train.max(axis=0).toarray().ravel()
sorted_by_tfidf = max_value.argsort()
feature_names = np.array(vectorizer.get_feature_names_out())
print("Features with lowest tfidf:\n{}".format(
feature_names[sorted_by_tfidf[:20]]))
print("Features with highest tfidf: \n{}".format(
feature_names[sorted_by_tfidf[-20:]]))

Features with lowest tfidf:
['service' 'my' 'but' 'have' 'at' 'here' 'time' 'go' 'an' 'one' 'don'
 'just' 'out' 'your' 'delicious' 'only' 'nice' 'would' 'came' 'ever']
Features with highest tfidf: 
['hot' 'she' 'burger' 'slow' 'sandwich' 'probably' 'sauce' 'flavor'
 'dishes' 'meal' 'selection' 'dining' 'you' 'another' 'area' 'once'
 'steak' 'wait' 'waited' 'said']


Аналіз показників IDF (Inverse Document Frequency)

Показник **IDF** дозволяє оцінити інформативність слів. Чим нижче значення IDF, тим частіше слово зустрічається в усіх відгуках корпусу. Це допомагає ідентифікувати "стоп-слова" специфічні для конкретного набору даних (наприклад, слова "food" або "place", які є в більшості відгуків ресторанів).

Сортування ознак за зростанням IDF дозволяє побачити найбільш загальні терміни, які мають найменшу вагу при класифікації.

In [25]:
sorted_by_idf = np.argsort(vectorizer.idf_)
print("Features with lowest idf:\n{}".format(
feature_names[sorted_by_idf[:100]]))

Features with lowest idf:
['the' 'and' 'was' 'to' 'is' 'it' 'this' 'of' 'food' 'not' 'in' 'place'
 'for' 'good' 'we' 'service' 'had' 'my' 'great' 'so' 'that' 'with' 'be'
 'were' 'back' 'but' 'very' 'have' 'at' 'you' 'are' 'here' 'on' 'they'
 'like' 'all' 'time' 'our' 'really' 'will' 'there' 'go' 'best' 'as' 'an'
 'one' 'don' 'their' 'just' 'out' 'delicious' 'your' 'only' 'would' 'nice'
 'up' 'friendly' 'came' 'also' 'did' 'ever' 'by' 'again' 'restaurant'
 'can' 'no' 'us' 'vegas' 'when' 'if' 'amazing' 'what' 'first' 'love'
 'pretty' 'won' 'experience' 'from' 'salad' 'going' 'never' 'disappointed'
 'even' 'too' 'been' 'say' 'got' 'he' 'which' 'staff' 'get' 'me' 'much'
 'about' 'bad' 'eat' 'definitely' 'because' 'always' 'more']


In [26]:
cv = CountVectorizer(ngram_range=(1, 1)).fit(x_train)
print("Vocabulary size: {}".format(len(cv.vocabulary_)))
print("Vocabulary:\n{}".format(cv.get_feature_names_out()))

Vocabulary size: 1728
Vocabulary:
['10' '100' '12' ... 'yum' 'yummy' 'zero']


In [27]:
pipee = make_pipeline(TfidfVectorizer(min_df=5), LogisticRegression())
paramm_grid = {"logisticregression__C": [0.001, 0.01, 0.1, 1, 10, 100],
"tfidfvectorizer__ngram_range": [(1, 1), (1, 2), (1, 3)]}

gridd = GridSearchCV(pipee, paramm_grid, cv=5)
gridd.fit(x_train, y_train)
print("Best cross-validation score: {:.2f}".format(gridd.best_score_))
print("Best parameters:\n{}".format(gridd.best_params_))

Best cross-validation score: 0.77
Best parameters:
{'logisticregression__C': 1, 'tfidfvectorizer__ngram_range': (1, 1)}


In [28]:

y_pred_tfidf = gridd.best_estimator_.predict(x_test)
print("TF-IDF")
print(classification_report(y_test, y_pred_tfidf))

TF-IDF
              precision    recall  f1-score   support

           0       0.78      0.81      0.79       128
           1       0.79      0.75      0.77       122

    accuracy                           0.78       250
   macro avg       0.78      0.78      0.78       250
weighted avg       0.78      0.78      0.78       250



### Word2Vec

In [29]:
def tokenizer(text):
    tokens = word_tokenize(text.lower())
    return [t for t in tokens if t.isalpha()]

x_train_tokens = [tokenizer(text) for text in x_train]
x_test_tokens = [tokenizer(text) for text in x_test]

In [30]:
model = Word2Vec(sentences=x_train_tokens, vector_size=100, window=5, min_count=1, workers=4)


In [31]:
word_vectors = model.wv

In [32]:

def get_avg_vector(tokens, model, size):
    vec = np.zeros(size)
    count = 0
    for word in tokens:
        if word in model.wv:
            vec += model.wv[word]
            count += 1
    return vec / count if count > 0 else vec

X_train_w2v = np.array([get_avg_vector(t, model, 100) for t in x_train_tokens])
X_test_w2v  = np.array([get_avg_vector(t, model, 100) for t in x_test_tokens])


In [33]:
grid_w2v = GridSearchCV(LogisticRegression(max_iter=1000), {'C': [0.1, 1, 10, 100]}, cv=5)
grid_w2v.fit(X_train_w2v, y_train)
print("Точність Word2Vec:", grid_w2v.best_score_)

Точність Word2Vec: 0.6066666666666667


In [34]:

y_pred = grid_w2v.predict(X_test_w2v)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.67      0.48      0.56       128
           1       0.58      0.75      0.65       122

    accuracy                           0.61       250
   macro avg       0.62      0.62      0.61       250
weighted avg       0.62      0.61      0.61       250



In [35]:
print("Порівняння методів")
print(f"BoW (baseline):    {np.mean(scores):.2f}")
print(f"TF-IDF + n-grams:  {gridd.best_score_:.2f}")
print(f"Word2Vec:          {grid_w2v.best_score_:.2f}")

Порівняння методів
BoW (baseline):    0.82
TF-IDF + n-grams:  0.77
Word2Vec:          0.61
